<a href="https://colab.research.google.com/github/Yolhe/-surrogate-nas-histology/blob/main/delfin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Kernel
 — es una pequeña matriz (3×3, 5×5, 7×7) que se desliza sobre la imagen buscando un patrón. Tamaño 3 detecta detalles finos, tamaño 7 detecta patrones más grandes. El algoritmo decide qué tamaño usar en cada capa.
###Filtros
— cuántos kernels diferentes aplicas en cada capa. Si usas 64 filtros, obtienes 64 versiones distintas de la imagen, cada una detectando algo diferente. Más filtros = más capacidad pero más lento.
##FC (Fully Connected)
— al final de las capas conv, los datos se aplanan y pasan por una capa densa normal, como en una red neuronal básica. El número aquí es cuántas neuronas tiene esa capa antes de la clasificación final.
##Dropout
 — durante el entrenamiento, apaga aleatoriamente un porcentaje de neuronas. Sirve para que la red no memorice los datos sino que aprenda patrones reales. 0.3 significa que apaga el 30% en cada paso.
##Pooling
 — después de cada capa conv, reduce el tamaño de la imagen a la mitad. MaxPool se queda con el valor máximo de cada zona, AvgPool saca el promedio. Sirve para reducir cómputo y hacer la red más robusta.

In [1]:
!pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
import numpy as np

In [3]:
SEARCH_SPACE = {
    'kernels':  [3, 5, 7],
    'filters':  [32, 64, 128, 256],
    'fc':       [128, 256, 512],
    'dropout':  [0.1, 0.2, 0.3, 0.4, 0.5],
    'pooling':  [0, 1, 2]
}

In [4]:
def get_pool(tipo):
    if tipo == 0:
        return nn.MaxPool2d(2, 2) #Divide la imagen en bloques de $2 \times 2$ píxeles y se queda solo con el valor más alto (el máximo) de cada bloque.
    elif tipo == 1:
        return nn.AvgPool2d(2, 2) #Divide la imagen en bloques de $2 \times 2$ píxeles, pero en lugar del máximo, calcula el promedio (Average) de los 4 píxeles.
    else:
        return nn.Identity() #Es un marcador de posición. No hace absolutamente nada; deja pasar la imagen exactamente igual como entró, sin alterar su tamaño ni sus valores.

In [5]:
def vector_a_cnn(vector, num_clases, input_channels=3):

    # Paso 1: decodificar el vector, es como un espacio de variables
    k1 = SEARCH_SPACE['kernels'][vector[0]]
    f1 = SEARCH_SPACE['filters'][vector[1]]
    k2 = SEARCH_SPACE['kernels'][vector[2]]
    f2 = SEARCH_SPACE['filters'][vector[3]]
    k3 = SEARCH_SPACE['kernels'][vector[4]]
    f3 = SEARCH_SPACE['filters'][vector[5]]
    fc = SEARCH_SPACE['fc'][vector[6]]
    dr = SEARCH_SPACE['dropout'][vector[7]]
    pl = SEARCH_SPACE['pooling'][vector[8]]

    # Paso 2: construir la red
    modelo = nn.Sequential(

        # Bloque 1
        nn.Conv2d(input_channels, f1, kernel_size=k1, padding=k1//2),
        nn.BatchNorm2d(f1),
        nn.ReLU(),
        get_pool(pl),

        # Bloque 2
        nn.Conv2d(f1, f2, kernel_size=k2, padding=k2//2),
        nn.BatchNorm2d(f2),
        nn.ReLU(),
        get_pool(pl),

        # Bloque 3
        nn.Conv2d(f2, f3, kernel_size=k3, padding=k3//2),
        nn.BatchNorm2d(f3),
        nn.ReLU(),
        get_pool(pl),

        # Clasificador final
        nn.AdaptiveAvgPool2d((4, 4)),
        nn.Flatten(),
        nn.Linear(f3 * 4 * 4, fc),
        nn.ReLU(),
        nn.Dropout(dr),
        nn.Linear(fc, num_clases)
    )

    return modelo

In [6]:
# Creamos un vector de prueba
vector_prueba = [0, 1, 2, 0, 1, 2, 1, 2, 0]

# Construimos la CNN con 10 clases
modelo = vector_a_cnn(vector_prueba, num_clases=10)

# Creamos una imagen falsa para verificar que la red funciona
# 1 imagen, 3 canales RGB, 64x64 píxeles
imagen_falsa = torch.randn(1, 3, 64, 64)

# Pasamos la imagen por la red
salida = modelo(imagen_falsa)

print("La red funciona correctamente")
print(f"Forma de la salida: {salida.shape}")

La red funciona correctamente
Forma de la salida: torch.Size([1, 10])


#bloque para dataset de kaggle

In [7]:
from google.colab import files
files.upload()  # selecciona el kaggle.json que descargaste

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"joseeloy","key":"b474754abb1c40d82927d56d6bc39478"}'}

In [8]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

!kaggle datasets download -d colorectal-histology-mnist/colorectal-histology-mnist

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [9]:
!unzip -q colorectal-histology-mnist.zip -d /content/nct_crc
!ls /content/nct_crc

unzip:  cannot find or open colorectal-histology-mnist.zip, colorectal-histology-mnist.zip.zip or colorectal-histology-mnist.zip.ZIP.
ls: cannot access '/content/nct_crc': No such file or directory


In [10]:
import os

for root, dirs, files in os.walk('/content/nct_crc'):
    level = root.replace('/content/nct_crc', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # solo muestra 2 niveles para no saturar
        for file in files[:3]:  # solo 3 archivos de muestra
            print(f'{indent}  {file}')

In [11]:
!pip install kaggle --upgrade -q
!kaggle datasets list -s "colorectal histology"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 26.4 MB/s eta 0:00:00
ref                                                                  title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
kmader/colorectal-histology-mnist                                    Colorectal Histology MNIST                           2042743521  2018-09-19 14:20:49.237000           7745        165  0.64705884       
abdurrahmanrocky/colorectal-histology                                Colorectal Histology                                   22239543  2021-10-22 03:30:37.030000             50          2  0.29411766       
sani84/glasmic

In [12]:
!kaggle datasets download -d kmader/colorectal-histology-mnist
!unzip -q colorectal-histology-mnist.zip -d /content/nct_crc
!ls /content/nct_crc

Dataset URL: https://www.kaggle.com/datasets/kmader/colorectal-histology-mnist
License(s): unknown
100% 1.90G/1.90G [01:33<00:00, 21.9MB/s]

hmnist_28_28_L.csv    kather_texture_2016_image_tiles_5000
hmnist_28_28_RGB.csv  Kather_texture_2016_image_tiles_5000
hmnist_64_64_L.csv    kather_texture_2016_larger_images_10
hmnist_8_8_L.csv      Kather_texture_2016_larger_images_10
hmnist_8_8_RGB.csv


In [13]:
!ls /content/nct_crc/kather_texture_2016_image_tiles_5000/Kather_texture_2016_image_tiles_5000
# de aqui salen la cantidad de las clases

01_TUMOR   03_COMPLEX  05_DEBRIS  07_ADIPOSE
02_STROMA  04_LYMPHO   06_MUCOSA  08_EMPTY


#empezar a usar el dataset

In [14]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Ruta al dataset
DATASET_PATH = '/content/nct_crc/kather_texture_2016_image_tiles_5000/Kather_texture_2016_image_tiles_5000'
NUM_CLASES = 8

# Transformaciones — prepara las imágenes para la CNN
transform = transforms.Compose([
    transforms.Resize((64, 64)),       # todas al mismo tamaño
    transforms.ToTensor(),             # convierte a tensor
    transforms.Normalize(              # normaliza los valores
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Cargar el dataset completo
dataset_completo = ImageFolder(root=DATASET_PATH, transform=transform)

# Dividir en entrenamiento (80%) y validación (20%)
total = len(dataset_completo)
val_size = int(total * 0.2)
train_size = total - val_size

train_dataset, val_dataset = random_split(dataset_completo, [train_size, val_size])

# Crear los DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Total imágenes: {total}")
print(f"Entrenamiento: {train_size}")
print(f"Validación: {val_size}")
print(f"Clases: {dataset_completo.classes}")

Total imágenes: 5000
Entrenamiento: 4000
Validación: 1000
Clases: ['01_TUMOR', '02_STROMA', '03_COMPLEX', '04_LYMPHO', '05_DEBRIS', '06_MUCOSA', '07_ADIPOSE', '08_EMPTY']


In [15]:
def evaluar_arquitectura(vector, epocas=3):
    """
    Recibe un vector, construye la CNN, la entrena brevemente
    y regresa el accuracy de validación como fitness.

    epocas=3 porque solo queremos estimar qué tan buena es
    la arquitectura, no entrenarla completo.
    """

    # Detectar si hay GPU disponible
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Construir la CNN con el vector
    modelo = vector_a_cnn(vector, num_clases=NUM_CLASES)
    modelo = modelo.to(device) #mandarlo a gpu

    # Optimizador y función de pérdida
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
    criterio    = nn.CrossEntropyLoss()

    # ── ENTRENAMIENTO ──
    modelo.train()
    for epoca in range(epocas):
        for imagenes, etiquetas in train_loader:
            imagenes = imagenes.to(device)
            etiquetas = etiquetas.to(device)

            optimizador.zero_grad()        # limpiar gradientes
            salida = modelo(imagenes)      # pasar imágenes por la red
            perdida = criterio(salida, etiquetas)  # calcular error
            perdida.backward()             # retropropagación
            optimizador.step()             # actualizar pesos

    # ── VALIDACIÓN ──
    modelo.eval()
    correctas = 0
    total = 0

    with torch.no_grad():  # no calcular gradientes en validación
        for imagenes, etiquetas in val_loader:
            imagenes = imagenes.to(device)
            etiquetas = etiquetas.to(device)

            salida = modelo(imagenes)
            _, predicciones = torch.max(salida, 1)  # clase con mayor probabilidad

            correctas += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)

    accuracy = correctas / total
    return accuracy

In [16]:
import torch
print(torch.cuda.is_available())        # debe imprimir True
print(torch.cuda.get_device_name(0))    # debe imprimir el nombre de la GPU

True
Tesla T4


In [17]:
# Prueba rápida — debe tardar unos minutos
vector_prueba = [0, 1, 0, 1, 0, 1, 1, 2, 0]
accuracy = evaluar_arquitectura(vector_prueba, epocas=1)
print(f"Accuracy de prueba: {accuracy:.4f}")

Accuracy de prueba: 0.6140


In [18]:
class SurrogateModel(nn.Module):
    def __init__(self, input_dim=9):
        super(SurrogateModel, self).__init__()

        self.red = nn.Sequential(
            nn.Linear(input_dim, 32),  # entrada: 9 genes del vector
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),          # salida: un número (accuracy predicho)
            nn.Sigmoid()               # entre 0 y 1
        )

    def forward(self, x):
        return self.red(x)


class GestorSurrogate:
    """
    Maneja el entrenamiento y uso del surrogate model.
    """
    def __init__(self):
        self.modelo   = SurrogateModel()
        self.optimizador = torch.optim.Adam(self.modelo.parameters(), lr=0.001)
        self.criterio = nn.MSELoss()  # error cuadrático medio

        # Memoria: aquí guardamos los vectores y sus accuracy reales
        self.vectores_vistos  = []
        self.accuracies_reales = []

    def agregar_resultado(self, vector, accuracy):
        """Guarda un vector y su accuracy real para entrenar el surrogate."""
        self.vectores_vistos.append(vector)
        self.accuracies_reales.append(accuracy)

    def entrenar(self, epocas=50):
        """Entrena el surrogate con los datos acumulados."""

        if len(self.vectores_vistos) < 5:
            return  # necesitamos al menos 5 ejemplos

        X = torch.tensor(self.vectores_vistos, dtype=torch.float32)
        y = torch.tensor(self.accuracies_reales, dtype=torch.float32).unsqueeze(1)

        self.modelo.train()
        for _ in range(epocas):
            self.optimizador.zero_grad()
            prediccion = self.modelo(X)
            perdida = self.criterio(prediccion, y)
            perdida.backward()
            self.optimizador.step()

    def predecir(self, vector):
        """Predice el accuracy de un vector sin entrenar la CNN."""
        self.modelo.eval()
        with torch.no_grad():
            x = torch.tensor(vector, dtype=torch.float32).unsqueeze(0)
            return self.modelo(x).item()

    def listo(self):
        """Regresa True cuando hay suficientes datos para confiar en el surrogate."""
        return len(self.vectores_vistos) >= 10

In [19]:
# Crear el gestor
gestor = GestorSurrogate()

# Simular que ya evaluamos 10 arquitecturas de verdad
import random

opciones = [3, 4, 3, 4, 3, 4, 3, 5, 3]  # tamaño de cada gen

for _ in range(10):
    vector_random = [random.randint(0, opciones[i]-1) for i in range(9)]
    accuracy_simulado = random.uniform(0.5, 0.9)
    gestor.agregar_resultado(vector_random, accuracy_simulado)

# Entrenar el surrogate con esos datos
gestor.entrenar(epocas=100)

# Predecir un vector nuevo
vector_nuevo = [0, 1, 2, 0, 1, 2, 1, 2, 0]
prediccion = gestor.predecir(vector_nuevo)
print(f"Accuracy predicho por surrogate: {prediccion:.4f}")
print(f"Surrogate listo: {gestor.listo()}")

Accuracy predicho por surrogate: 0.8023
Surrogate listo: True


In [21]:
from scipy.stats import cauchy

class LSHADE():
    def __init__(self, tam_poblacion=20, H=10):
        """
        tam_poblacion: cuántos vectores en la población
        H: tamaño del archivo histórico de F y CR exitosos
        """
        self.tam_poblacion = tam_poblacion
        self.H = H

        # Archivo histórico — guarda F y CR que funcionaron bien
        self.M_F  = [0.5] * H   # historial de F
        self.M_CR = [0.5] * H   # historial de CR
        self.k = 0               # índice actual del historial

        # Archivo de soluciones descartadas (para mutación)
        self.archivo = []

        # Límites de cada gen
        self.limites = [
            (0, 2),  # k1 — kernels tiene 3 opciones (índices 0,1,2)
            (0, 3),  # f1 — filters tiene 4 opciones
            (0, 2),  # k2
            (0, 3),  # f2
            (0, 2),  # k3
            (0, 3),  # f3
            (0, 2),  # fc — tiene 3 opciones
            (0, 4),  # dropout — tiene 5 opciones
            (0, 2),  # pooling — tiene 3 opciones
        ]

        # Inicializar población aleatoria
        self.poblacion = self._poblacion_inicial()
        self.fitness   = [None] * tam_poblacion

    def _poblacion_inicial(self):
        """Genera vectores aleatorios respetando los límites de cada gen."""
        poblacion = []
        for _ in range(self.tam_poblacion):
            vector = [random.randint(lim[0], lim[1])
                      for lim in self.limites]
            poblacion.append(vector)
        return poblacion

    def _generar_F_CR(self):
        """Genera F y CR a partir del historial."""
        idx = random.randint(0, self.H - 1)

        # F desde distribución de Cauchy
        F = cauchy.rvs(loc=self.M_F[idx], scale=0.1)
        F = float(np.clip(F, 0.1, 1.0))

        # CR desde distribución normal
        CR = np.random.normal(self.M_CR[idx], 0.1)
        CR = float(np.clip(CR, 0.0, 1.0))

        return F, CR

    def _mutar(self, idx, F):
        """
        Crea un vector mutado usando DE/current-to-pbest/1.
        Combina el mejor conocido con diferencias de otros vectores.
        """
        # Seleccionar el mejor 20% de la población
        p = max(2, int(0.2 * self.tam_poblacion))
        fitness_validos = [(i, f) for i, f in enumerate(self.fitness) if f is not None]

        if len(fitness_validos) < 3:
            # Si no hay suficientes evaluados, mutación aleatoria
            candidatos = [i for i in range(self.tam_poblacion) if i != idx]
            a, b = random.sample(candidatos, 2)
            mutado = [
                self.poblacion[idx][g] + F * (self.poblacion[a][g] - self.poblacion[b][g])
                for g in range(9)
            ]
        else:
            # Seleccionar pbest — uno de los mejores
            fitness_validos.sort(key=lambda x: x[1], reverse=True)
            pbest_idx = random.choice([i for i, _ in fitness_validos[:p]])

            # Seleccionar r1 diferente de idx y pbest
            candidatos = [i for i in range(self.tam_poblacion) if i != idx and i != pbest_idx]
            r1 = random.choice(candidatos)

            # r2 puede venir del archivo también
            pool = self.poblacion + self.archivo
            r2_vector = random.choice([v for i, v in enumerate(pool)
                                       if i != idx and i != pbest_idx and i != r1])

            mutado = [
                self.poblacion[idx][g] + F * (self.poblacion[pbest_idx][g] - self.poblacion[idx][g])
                + F * (self.poblacion[r1][g] - r2_vector[g])
                for g in range(9)
            ]

        # Redondear y aplicar límites
        mutado = [int(round(np.clip(mutado[g], self.limites[g][0], self.limites[g][1])))
                  for g in range(9)]

        return mutado

    def _cruzar(self, original, mutado, CR):
        """Mezcla genes del original y el mutado según CR."""
        trial = []
        j_rand = random.randint(0, 8)  # al menos un gen viene del mutado

        for g in range(9):
            if random.random() < CR or g == j_rand:
                trial.append(mutado[g])
            else:
                trial.append(original[g])

        return trial

    def _actualizar_historial(self, F_exitosos, CR_exitosos):
        """Actualiza M_F y M_CR con los valores que funcionaron bien."""
        if not F_exitosos:
            return

        # Media de Lehmer para F
        self.M_F[self.k] = sum(f**2 for f in F_exitosos) / sum(F_exitosos)
        # Media aritmética para CR
        self.M_CR[self.k] = sum(CR_exitosos) / len(CR_exitosos)

        self.k = (self.k + 1) % self.H

    def _reducir_poblacion(self, generacion, max_generaciones):
        """Reduce el tamaño de la población linealmente — característica de L-SHADE."""
        N_min = 4
        N_init = self.tam_poblacion
        nuevo_tam = max(N_min, int(N_init - (N_init - N_min) * generacion / max_generaciones))

        if nuevo_tam < len(self.poblacion):
            # Eliminar los peores
            indices_ordenados = sorted(
                range(len(self.poblacion)),
                key=lambda i: self.fitness[i] if self.fitness[i] is not None else -1,
                reverse=True
            )
            self.poblacion = [self.poblacion[i] for i in indices_ordenados[:nuevo_tam]]
            self.fitness   = [self.fitness[i]   for i in indices_ordenados[:nuevo_tam]]

    def paso(self, generacion, max_generaciones, gestor_surrogate, evaluar_real):
        """
        Ejecuta una generación completa de L-SHADE.
        Usa surrogate para evaluar rápido, solo evalúa de verdad
        los más prometedores.
        """
        F_exitosos  = []
        CR_exitosos = []

        nuevos_vectores = []
        nuevos_F  = []
        nuevos_CR = []

        for i in range(len(self.poblacion)):
            F, CR = self._generar_F_CR()
            mutado = self._mutar(i, F)
            trial  = self._cruzar(self.poblacion[i], mutado, CR)

            nuevos_vectores.append(trial)
            nuevos_F.append(F)
            nuevos_CR.append(CR)

        # Evaluar con surrogate si está listo, si no evaluar de verdad
        for i, trial in enumerate(nuevos_vectores):
            if gestor_surrogate.listo():
                fitness_trial = gestor_surrogate.predecir(trial)
            else:
                fitness_trial = evaluar_real(trial)
                gestor_surrogate.agregar_resultado(trial, fitness_trial)

            # Evaluar el original si no tiene fitness todavía
            if self.fitness[i] is None:
                self.fitness[i] = evaluar_real(self.poblacion[i])
                gestor_surrogate.agregar_resultado(self.poblacion[i], self.fitness[i])

            # Selección — sobrevive el mejor
            if fitness_trial >= self.fitness[i]:
                self.archivo.append(self.poblacion[i])
                self.poblacion[i] = trial
                self.fitness[i]   = fitness_trial
                F_exitosos.append(nuevos_F[i])
                CR_exitosos.append(nuevos_CR[i])

        # Limitar tamaño del archivo
        if len(self.archivo) > len(self.poblacion):
            self.archivo = random.sample(self.archivo, len(self.poblacion))

        # Actualizar historial y reducir población
        self._actualizar_historial(F_exitosos, CR_exitosos)
        self._reducir_poblacion(generacion, max_generaciones)

        # Regresar el mejor de esta generación
        mejor_idx = max(range(len(self.fitness)),
                        key=lambda i: self.fitness[i] if self.fitness[i] is not None else -1)
        return self.poblacion[mejor_idx], self.fitness[mejor_idx]
